# CoDaS on All of Us: free biomarker exploration

Run this inside an All of Us Researcher Workbench 2.0 workspace (JupyterLab app), top to bottom.

1. **Setup**: install CoDaS (`aou` branch) and point it at Vertex AI in this workspace's project.
2. **Checks**: the CDR has the Fitbit columns the spec needs; every catalog outcome's concept ids
   exist, sit in the right domain and have the expected names.
3. **Feasibility screen**: participants (and cases) per outcome; counts of 1-20 are masked.
4. **Exploration**: CoDaS runs on the top outcomes and writes masked reports to `.codas_runs/aou/`.

What the model sees: column names, aggregate statistics, and counts with 1-20 masked. Never a row.
The extracted tables stay on this workspace disk (git-ignored). Review every output against the
Data and Statistics Dissemination Policy before anything leaves the Workbench, and do not commit
`.codas_runs/`.

## 1. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (REPO / "codas" / "aou").exists(), f"Run this notebook from the codas-adk checkout (got {REPO})."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[aou]"], check=True)
sys.path.insert(0, str(REPO))

In [ ]:
# --- edit if needed (everything below is detected automatically) ------------------------------
CDR = os.getenv("CODAS_AOU_CDR") or os.getenv("WORKSPACE_CDR") or ""   # "<cdr-project>.<cdr-dataset>"
REASONING_MODEL = ""   # leave empty to auto-pick; or a Gemini id, or a Claude id enabled in Model Garden
FAST_MODEL = ""
TOP_K = 3              # outcomes to run CoDaS on after the screen
# ---------------------------------------------------------------------------------------------
import json, re

def sh(*cmd):
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

project = os.getenv("GOOGLE_CLOUD_PROJECT") or sh("gcloud", "config", "get-value", "project")
assert project, "No Google Cloud project found; set GOOGLE_CLOUD_PROJECT to this workspace's project."

def find_cdr():
    """The Registered/Controlled Tier CDR among the workspace's BigQuery resources (wb resource list)."""
    try:
        resources = json.loads(sh("wb", "resource", "list", "--format=json") or "[]")
    except json.JSONDecodeError:
        return ""
    def walk(obj):
        if isinstance(obj, dict):
            ds = obj.get("datasetId") or obj.get("dataset_id")
            proj = obj.get("projectId") or obj.get("project_id")
            if isinstance(ds, str) and isinstance(proj, str) and re.fullmatch(r"[RC]\d{4}Q\dR\d+", ds):
                yield f"{proj}.{ds}"
            for value in obj.values():
                yield from walk(value)
        elif isinstance(obj, list):
            for value in obj:
                yield from walk(value)
    found = sorted(set(walk(resources)))
    return found[0] if found else ""

CDR = CDR or find_cdr()
assert CDR, ("Could not find the CDR. Open the workspace's Resources tab, click the R2025Q4R6 dataset, and set "
             "CDR = '<Project ID>.<Dataset ID>' at the top of this cell.")

# Pick a Vertex location and models that answer from this project. Every attempt is logged, so a
# failure says why (404 = model not offered there, 403 = permission or perimeter).
import warnings
from google import genai
warnings.filterwarnings("ignore", message=".*automatic function calling.*")

REASONING_CANDIDATES = ["gemini-3.1-pro-preview", "gemini-3-pro-preview", "gemini-pro-latest", "gemini-2.5-pro"]
FAST_CANDIDATES = ["gemini-3.5-flash", "gemini-3-flash-preview", "gemini-flash-latest", "gemini-2.5-flash"]
LOCATIONS = ["global", "us-central1", "us-east4", "us-west1"]
attempts = []

def answers(location, model):
    try:
        genai.Client(vertexai=True, project=project, location=location).models.generate_content(
            model=model, contents="Say OK")
        attempts.append((location, model, "OK"))
        return True
    except Exception as exc:
        attempts.append((location, model, f"{type(exc).__name__}: {str(exc)[:110]}"))
        return False

def first_working(location, preferred, candidates):
    if preferred.lower().startswith("claude"):
        return preferred   # Claude on Vertex is not probed through the Gemini client
    return next((m for m in dict.fromkeys([preferred, *candidates]) if m and answers(location, m)), None)

chosen = None
for location in LOCATIONS:
    fast = first_working(location, FAST_MODEL, FAST_CANDIDATES)
    if not fast:
        continue
    reasoning = first_working(location, REASONING_MODEL, REASONING_CANDIDATES) or fast
    chosen = (location, reasoning, fast)
    break
if not chosen:
    for row in attempts:
        print(" | ".join(row))
    raise SystemExit("No Gemini model answered from this project. Copy the table above (it holds no participant "
                     "data) and share it to decide the fix.")
location, REASONING_MODEL, FAST_MODEL = chosen

os.environ.update({
    "CODAS_AOU_MODE": "1",
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "GOOGLE_CLOUD_PROJECT": project,
    "GOOGLE_CLOUD_LOCATION": location,
    "CODAS_AOU_CDR": CDR,
    "CODAS_GEMINI_REASONING_MODEL": REASONING_MODEL,
    "CODAS_GEMINI_FAST_MODEL": FAST_MODEL,
    "CODAS_GEMINI_MODELS": ",".join(dict.fromkeys([FAST_MODEL, REASONING_MODEL])),  # literature search
})
for key in ("GOOGLE_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY"):
    os.environ.pop(key, None)   # AoU mode refuses consumer API keys

from codas.aou.privacy import check_llm_backend
check_llm_backend()
print(f"project:  {project}\nCDR:      {CDR}\nlocation: {location}\nmodels:   {REASONING_MODEL} (reasoning), {FAST_MODEL} (fast)")
print("model backend: Vertex AI in the workspace project (OK)")

## 2. Checks: CDR schema and outcome catalog

In [ ]:
from google.cloud import bigquery
from IPython.display import Markdown, display

from codas.aou import cohort, feasibility as feas, runner
from codas.aou.cohort import CohortSpec

client = bigquery.Client(project=project)
BASE = CohortSpec.from_json(REPO / "scripts/aou/hba1c_fitbit.example.json")   # wearables, window, demographics
CATALOG = REPO / "scripts/aou/outcomes_catalog.json"

present = cohort._query(client, *cohort.schema_check_sql(BASE, CDR))
missing = cohort.missing_columns(BASE, list(zip(present["table_name"], present["column_name"])))
if missing:
    print("Missing in this CDR:", ", ".join(missing))
    print("Columns that do exist in those tables:")
    for table, cols in present.groupby("table_name")["column_name"]:
        print(f"  {table}: {', '.join(sorted(cols))}")
    raise SystemExit("Edit scripts/aou/hba1c_fitbit.example.json to match, then re-run this cell.")
print("wearable columns: all present")

outcomes = feas.load_catalog(CATALOG)
problems = feas.verify_catalog(outcomes, CDR, client)
display(Markdown("| outcome | concept check |\n|---|---|\n" + "\n".join(
    f"| {o.label} | {problems[o.name] or 'ok'} |" for o in outcomes)))

## 3. Feasibility screen (counts only; 1-20 masked)

In [ ]:
screen = runner.explore(CATALOG, BASE, CDR, client=client, top_k=TOP_K, screen_only=True, log=lambda *_: None)
display(Markdown(screen["table"]))
print("selected for CoDaS:", screen["chosen"] or "none (lower the minimums in runner.explore if needed)")

## 4. Exploration

Runs CoDaS (extraction, profiling, the discovery loop, critic/defender debate, reporting) on each
selected outcome. Expect several minutes per outcome. Only tool names are printed while it runs.

In [ ]:
def progress(agent, tool):
    print(f"  {agent:24s} -> {tool}()")

result = runner.explore(CATALOG, BASE, CDR, client=client, top_k=TOP_K, on_tool_call=progress)
display(Markdown(Path(result["summary_path"]).read_text()))

In [ ]:
for run in result["runs"]:
    display(Markdown(f"---\n## {run['cohort']}\n\n" + Path(run["report_path"]).read_text()))

## Before anything leaves the Workbench

- Outputs are in `.codas_runs/aou/`: per-outcome `*_report.md`, `*_rounds.json`, and `explore_summary.md`.
- Counts of 1-20 are masked by CoDaS, but check every table and figure yourself against the
  Dissemination Policy (including counts that could be derived by subtraction).
- The `*.csv` files are participant-level. Never download, copy, commit or paste them anywhere.